[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emersonnjsantos/visao_computacional/blob/main/ATIVIDADE_5/relatorio_atividade_5_projeto.ipynb)


# Atividade 5 — Análise Global, Local e de Textura em Mini Dataset Próprio
## Unidade 3 | Capítulo 2 | Tarefa 5

**Programa:** Residência em TIC43 — Módulo de Visão Computacional (Softex / MCTI)
**Aluno:** Emerson Noe Jose dos Santos
**Docentes:** Prof. Rafael Carmo | Prof. Alyson Bezerra | Prof. Matheus Araújo
**Repositório do Projeto:** [github.com/emersonnjsantos/visao_computacional](https://github.com/emersonnjsantos/visao_computacional)

**Objetivo:** Analisar tecnicamente o mini dataset próprio em três níveis complementares — global (histogramas e estatísticas), local (pontos de interesse, descritores ORB e matching) e textura (GLCM e LBP) —, comparando criticamente os descritores e justificando qual representação é mais adequada para distinguir as classes.


In [ ]:
# Instalação de dependências no Google Colab
!pip install scikit-image -q


In [ ]:
!git clone https://github.com/emersonnjsantos/visao_computacional.git 2>/dev/null || echo "Repositório já clonado." 


In [ ]:
# Importações e configurações globais
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
from scipy.stats import entropy as scipy_entropy
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.feature import blob_log
from skimage import img_as_ubyte

# Configurações do Matplotlib
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'

# Caminhos do dataset
DATASET_PATH = '/content/visao_computacional/ATIVIDADE_1/visao_computacional_mini_dataset/dataset'
CLASSE_A_PATH = os.path.join(DATASET_PATH, 'classe_A')
CLASSE_B_PATH = os.path.join(DATASET_PATH, 'classe_B')

# Carregamento de todas as imagens
imgs_A = sorted(glob.glob(os.path.join(CLASSE_A_PATH, '*.jpg')))
imgs_B = sorted(glob.glob(os.path.join(CLASSE_B_PATH, '*.jpg')))
todas_imagens = imgs_A + imgs_B

print("=" * 60)
print(" AMBIENTE DE VISÃO COMPUTACIONAL INICIALIZADO")
print("=" * 60)
print(f"• OpenCV:  {cv2.__version__}")
print(f"• NumPy:   {np.__version__}")
print(f"• Pandas:  {pd.__version__}")
print(f"• Classe A (Flores): {len(imgs_A)} imagens")
print(f"• Classe B (Folhas): {len(imgs_B)} imagens")
print(f"• Total:             {len(todas_imagens)} imagens")


---
## Parte 1 — Preparação do Dataset e Hipótese Inicial

### 1.1 Descrição das Classes

O mini dataset foi construído na **Atividade 1** e é composto por duas classes de imagens naturais:

| Classe | Conteúdo | Imagens | Descrição |
|--------|----------|---------|-----------|
| **Classe A** | Flores | `img001` a `img005` | Fotografias de flores com pétalas coloridas (vermelhas, rosas, amarelas), texturas suaves e formas arredondadas |
| **Classe B** | Folhas | `img006` a `img010` | Fotografias de folhas verdes com nervuras visíveis, texturas lineares e formas alongadas/serrilhadas |

**Dispositivo de captura:** Smartphone Xiaomi 11 Pro (sensor 108 MP, f/1.9)
**Condições:** Luz natural direta, luz difusa e luz artificial; distâncias de 15 cm a 1 m; ângulos variados.


### 1.2 Estrutura de Pastas

```
visao_computacional/
├── ATIVIDADE_1/
│   └── visao_computacional_mini_dataset/
│       ├── dataset/
│       │   ├── classe_A/          ← Flores (img001 a img005)
│       │   │   ├── img001_original.jpg
│       │   │   ├── img002_original.jpg
│       │   │   ├── img003_original.jpg
│       │   │   ├── img004_original.jpg
│       │   │   └── img005_original.jpg
│       │   └── classe_B/          ← Folhas (img006 a img010)
│       │       ├── img006_original.jpg
│       │       ├── img007_original.jpg
│       │       ├── img008_original.jpg
│       │       ├── img009_original.jpg
│       │       └── img010_original.jpg
│       └── README.md
├── ATIVIDADE_2/
├── ATIVIDADE_3/
├── ATIVIDADE_4/
└── ATIVIDADE_5/
    └── relatorio_atividade_5_projeto.ipynb  ← Este notebook
```


In [ ]:
# Exibição das imagens do dataset: 2 de cada classe
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Classe A
for i, idx in enumerate([0, 1]):
    img = cv2.imread(imgs_A[idx])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img_rgb)
    nome = os.path.basename(imgs_A[idx])
    axes[0, i].set_title(f'Classe A (Flores) — {nome}', fontsize=11)
    axes[0, i].axis('off')
    print(f"Classe A | {nome}: {img.shape[1]}x{img.shape[0]} px, {os.path.getsize(imgs_A[idx])//1024} KB")

# Classe B
for i, idx in enumerate([0, 1]):
    img = cv2.imread(imgs_B[idx])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img_rgb)
    nome = os.path.basename(imgs_B[idx])
    axes[1, i].set_title(f'Classe B (Folhas) — {nome}', fontsize=11)
    axes[1, i].axis('off')
    print(f"Classe B | {nome}: {img.shape[1]}x{img.shape[0]} px, {os.path.getsize(imgs_B[idx])//1024} KB")

plt.suptitle('Amostras Representativas do Mini Dataset', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


### 1.3 Hipótese Inicial

**Fatores de aquisição que podem influenciar a análise:**
- **Iluminação:** Variações entre sol direto, sombra e luz artificial afetam diretamente os histogramas de intensidade e cor, podendo mascarar diferenças entre classes.
- **Distância/Ângulo:** Close-ups de 15 cm revelam texturas finas (nervuras das folhas, texturas das pétalas), enquanto capturas a 1 m perdem esses microdetalhes.
- **Foco e profundidade de campo:** A abertura f/1.9 gera desfoque de fundo (bokeh), que pode afetar a detecção de keypoints nas bordas dos objetos.

**Diferenças visuais esperadas entre as classes:**
- **Cor:** Flores possuem cores vibrantes e variadas (vermelho, rosa, amarelo); folhas são predominantemente verdes, com menor variação cromática.
- **Textura:** Folhas apresentam nervuras lineares e paralelas bem definidas; flores possuem texturas mais suaves e homogêneas nas pétalas.
- **Forma:** Pétalas são arredondadas; folhas são mais alongadas com bordas serrilhadas.

**Hipótese sobre o descritor mais útil:**

> **Acredito que os histogramas de cor (análise global) serão o descritor mais discriminativo**, pois a diferença mais marcante entre flores e folhas reside na paleta cromática (flores multicoloridas vs. folhas predominantemente verdes). No entanto, espero que os descritores de textura (GLCM/LBP) também sejam informativos, principalmente para capturar as nervuras lineares das folhas.


---
## Parte 2 — Análise Global: Histogramas e Estatística

### 2.1 Estatística Global


In [ ]:
# Cálculo das estatísticas globais para as 10 imagens
def calcular_estatisticas(caminho):
    # Calcula média, variância, desvio padrão e entropia de uma imagem em escala de cinza.
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    media = np.mean(img)
    variancia = np.var(img)
    desvio = np.std(img)
    # Entropia de Shannon
    hist, _ = np.histogram(img.ravel(), bins=256, range=[0, 256], density=True)
    hist = hist[hist > 0]  # Remove bins vazios
    ent = scipy_entropy(hist, base=2)
    return media, variancia, desvio, ent

dados_estatisticas = []
for path in todas_imagens:
    nome = os.path.basename(path)
    classe = "A (Flores)" if path in imgs_A else "B (Folhas)"
    media, var, std, ent = calcular_estatisticas(path)
    dados_estatisticas.append({
        "Imagem": nome,
        "Classe": classe,
        "Média": round(media, 2),
        "Variância": round(var, 2),
        "Desvio Padrão": round(std, 2),
        "Entropia (bits)": round(ent, 3)
    })

df_stats = pd.DataFrame(dados_estatisticas)

print("=" * 80)
print("TABELA DE ESTATÍSTICAS GLOBAIS — 10 IMAGENS DO MINI DATASET")
print("=" * 80)
display(df_stats)

# Resumo por classe
resumo = df_stats.groupby("Classe")[["Média", "Variância", "Desvio Padrão", "Entropia (bits)"]].agg(["mean", "std"]).round(2)
print("\n" + "=" * 80)
print("RESUMO ESTATÍSTICO POR CLASSE")
print("=" * 80)
display(resumo)


### 2.2 Histogramas de Intensidade e Cor


In [ ]:
# Histogramas de intensidade (escala de cinza) — 2 imagens por classe
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

pares = [(imgs_A[0], imgs_A[1], "A (Flores)"), (imgs_B[0], imgs_B[1], "B (Folhas)")]

for row, (p1, p2, classe) in enumerate(pares):
    for col, p in enumerate([p1, p2]):
        img_gray = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        nome = os.path.basename(p)
        axes[row, col].hist(img_gray.ravel(), bins=256, range=[0, 256],
                            color='#2c3e50' if row == 0 else '#27ae60',
                            alpha=0.8, edgecolor='none')
        axes[row, col].set_title(f'Classe {classe} — {nome}')
        axes[row, col].set_xlabel('Nível de Intensidade')
        axes[row, col].set_ylabel('Frequência')
        axes[row, col].grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Histogramas de Intensidade (Escala de Cinza)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Histogramas de cor RGB — 2 imagens por classe
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
cores_rgb = ['#e74c3c', '#27ae60', '#3498db']
labels_rgb = ['R', 'G', 'B']

for row, (p1, p2, classe) in enumerate(pares):
    for col, p in enumerate([p1, p2]):
        img = cv2.imread(p)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        nome = os.path.basename(p)
        for ch in range(3):
            axes[row, col].hist(img_rgb[:, :, ch].ravel(), bins=256, range=[0, 256],
                                color=cores_rgb[ch], alpha=0.5, label=labels_rgb[ch],
                                edgecolor='none')
        axes[row, col].set_title(f'Classe {classe} — {nome}')
        axes[row, col].set_xlabel('Intensidade do Canal')
        axes[row, col].set_ylabel('Frequência')
        axes[row, col].legend()
        axes[row, col].grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Histogramas de Cor (Canais RGB)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Análise dos Histogramas

**Onde a imagem concentra mais pixels?**
- **Flores (Classe A):** Os histogramas de intensidade mostram distribuição bimodal ou com picos em tons médios a claros (120–200), correspondendo às pétalas iluminadas e ao fundo desfocado. Há componentes significativas no canal vermelho (R), refletindo as cores quentes das flores.
- **Folhas (Classe B):** Os histogramas concentram-se mais em tons médios (80–160), com forte dominância do canal verde (G), consistente com a clorofila presente nas folhas.

**Existe canal de cor dominante?**
- Sim. Nas flores, o canal R (vermelho) é claramente dominante em flores vermelhas/rosas, enquanto nas folhas o canal G (verde) predomina de forma consistente.

**As imagens da mesma classe apresentam comportamento parecido?**
- Sim, há um padrão intra-classe reconhecível: flores tendem a ter histogramas com maior dispersão cromática, enquanto folhas apresentam perfis mais concentrados no canal verde.

**O histograma ajudou a separar as classes?**
- **Sim**, especialmente os histogramas de cor RGB. A dominância do canal vermelho nas flores vs. verde nas folhas constitui uma assinatura discriminativa forte. Os histogramas de intensidade (cinza), por si só, são menos discriminativos devido à sobreposição de faixas tonais.


### 2.3 Relação com a Tarefa Anterior

Comparação entre a imagem original e uma versão derivada (JPEG com compressão forte e resolução reduzida) produzida na Atividade 1/2.


In [ ]:
# Gerar versões derivadas para comparação:
# 1. Redução de resolução a 20%
# 2. JPEG com compressão forte (quality=10)

def gerar_versao_derivada(img_path, fator_reducao=0.2, jpeg_quality=10):
    # Gera uma versão derivada com resolução reduzida e compressão JPEG.
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    new_w, new_h = int(w * fator_reducao), int(h * fator_reducao)
    img_reduzida = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    # Compressão JPEG em memória
    encode_params = [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality]
    _, buffer = cv2.imencode('.jpg', img_reduzida, encode_params)
    img_comprimida = cv2.imdecode(buffer, cv2.IMREAD_COLOR)
    return img_comprimida

# Classe A — img001
img_A_orig = cv2.imread(imgs_A[0], cv2.IMREAD_GRAYSCALE)
img_A_deriv = cv2.cvtColor(gerar_versao_derivada(imgs_A[0]), cv2.COLOR_BGR2GRAY)

# Classe B — img006
img_B_orig = cv2.imread(imgs_B[0], cv2.IMREAD_GRAYSCALE)
img_B_deriv = cv2.cvtColor(gerar_versao_derivada(imgs_B[0]), cv2.COLOR_BGR2GRAY)

# Calcular estatísticas de cada versão
def stats_resumo(img, nome):
    hist, _ = np.histogram(img.ravel(), 256, [0, 256], density=True)
    hist = hist[hist > 0]
    return {"Versão": nome, "Média": round(np.mean(img), 2),
            "Desvio": round(np.std(img), 2),
            "Entropia": round(scipy_entropy(hist, base=2), 3)}

tabela_comp = pd.DataFrame([
    stats_resumo(img_A_orig, "Classe A — Original"),
    stats_resumo(img_A_deriv, "Classe A — Derivada (20%, JPEG q=10)"),
    stats_resumo(img_B_orig, "Classe B — Original"),
    stats_resumo(img_B_deriv, "Classe B — Derivada (20%, JPEG q=10)")
])
print("COMPARAÇÃO: Original vs Versão Derivada")
display(tabela_comp)

# Histogramas comparativos
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(img_A_orig.ravel(), 256, [0, 256], color='#3498db', alpha=0.7, label='Original')
axes[0, 0].set_title('Classe A — Original (Histograma)')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.4)

axes[0, 1].hist(img_A_deriv.ravel(), 256, [0, 256], color='#e74c3c', alpha=0.7, label='Derivada')
axes[0, 1].set_title('Classe A — Derivada 20% + JPEG q=10')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.4)

axes[1, 0].hist(img_B_orig.ravel(), 256, [0, 256], color='#3498db', alpha=0.7, label='Original')
axes[1, 0].set_title('Classe B — Original (Histograma)')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.4)

axes[1, 1].hist(img_B_deriv.ravel(), 256, [0, 256], color='#e74c3c', alpha=0.7, label='Derivada')
axes[1, 1].set_title('Classe B — Derivada 20% + JPEG q=10')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.4)

plt.suptitle('Comparação de Histogramas: Original vs. Versão Derivada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Discussão da Comparação Original vs. Derivada

**O histograma mudou muito?**
- Sim. A versão derivada (resolução 20% + compressão JPEG com qualidade 10) apresenta um histograma com menos níveis de intensidade efetivamente ocupados e picos mais concentrados — efeito direto da quantização imposta pela compressão JPEG agressiva e pela redução de resolução que elimina texturas finas.

**A entropia caiu ou aumentou?**
- A entropia **caiu** na versão derivada. A compressão JPEG elimina detalhes de alta frequência e agrupa níveis de cinza vizinhos, reduzindo a diversidade de intensidades e, consequentemente, a entropia de Shannon. Isso indica que a versão comprimida carrega menos informação por pixel.

**Houve perda perceptível de informação útil?**
- Sim. A redução de resolução destrói microestruturas como nervuras de folhas e texturas de pétalas. A compressão JPEG introduz artefatos de bloco que degradam bordas e contornos. Essas perdas impactam diretamente descritores locais (ORB) e de textura (GLCM/LBP), embora os histogramas globais de cor mantenham parcialmente o padrão cromático de cada classe.


---
## Parte 3 — Análise Local: Pontos de Interesse, Descritores e Matching

### 3.1 Pontos de Interesse — Cantos de Shi-Tomasi e Blobs


In [ ]:
# Detecção de cantos Shi-Tomasi — 1 imagem de cada classe
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (path, classe) in enumerate([(imgs_A[0], "A (Flores)"), (imgs_B[0], "B (Folhas)")]):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detecção de cantos Shi-Tomasi
    corners = cv2.goodFeaturesToTrack(gray, maxCorners=200, qualityLevel=0.01, minDistance=10)
    img_corners = img_rgb.copy()
    if corners is not None:
        for c in corners:
            x, y = c.ravel()
            cv2.circle(img_corners, (int(x), int(y)), 5, (255, 0, 0), -1)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f'Classe {classe} — Original')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(img_corners)
    n_corners = len(corners) if corners is not None else 0
    axes[row, 1].set_title(f'Classe {classe} — Shi-Tomasi ({n_corners} cantos)')
    axes[row, 1].axis('off')

    print(f"Classe {classe}: {n_corners} cantos Shi-Tomasi detectados")

plt.suptitle('Detecção de Cantos — Shi-Tomasi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Detecção de blobs com blob_log — 1 imagem adicional de cada classe
from skimage.feature import blob_log
from skimage import img_as_float

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for col, (path, classe) in enumerate([(imgs_A[2], "A (Flores)"), (imgs_B[2], "B (Folhas)")]):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_float = img_as_float(gray)

    # Detecção de blobs (Laplacian of Gaussian)
    blobs = blob_log(gray_float, min_sigma=5, max_sigma=30, num_sigma=10, threshold=0.05)

    img_blobs = img_rgb.copy()
    for blob in blobs:
        y, x, sigma = blob
        r = int(sigma * np.sqrt(2))
        cv2.circle(img_blobs, (int(x), int(y)), r, (0, 255, 0), 2)

    axes[col].imshow(img_blobs)
    axes[col].set_title(f'Classe {classe} — blob_log ({len(blobs)} blobs)')
    axes[col].axis('off')

    print(f"Classe {classe}: {len(blobs)} blobs detectados")

plt.suptitle('Detecção de Blobs — Laplacian of Gaussian (LoG)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Interpretação dos Pontos de Interesse:**
- **Shi-Tomasi:** Nas flores, os cantos foram detectados predominantemente nas bordas das pétalas e nos estames centrais, onde há transições abruptas de cor e intensidade. Nas folhas, os cantos concentraram-se ao longo das nervuras e nas bordas serrilhadas, capturando a geometria linear da superfície.
- **Blobs (LoG):** Os blobs detectaram regiões com variação de escala como manchas e padrões circulares. Nas flores, capturam os centros das pétalas e estames; nas folhas, capturam pontos de convergência das nervuras. O método é complementar ao Shi-Tomasi, pois responde a regiões em vez de vértices.


### 3.2 Descritores Locais — ORB


In [ ]:
# Extração de keypoints e descritores ORB
# Par 1: mesma classe A (Flores — img001 vs img002)
# Par 2: mesma classe B (Folhas — img006 vs img007)
# Par 3: classes diferentes (Flores img001 vs Folhas img006)

orb = cv2.ORB_create(nfeatures=500)

pares_orb = [
    (imgs_A[0], imgs_A[1], "Mesmo Par — Classe A: Flores vs Flores"),
    (imgs_B[0], imgs_B[1], "Mesmo Par — Classe B: Folhas vs Folhas"),
    (imgs_A[0], imgs_B[0], "Par Diferente — Flores vs Folhas"),
]

resultados_orb = []

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for row, (p1, p2, titulo) in enumerate(pares_orb):
    img1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(p2, cv2.IMREAD_GRAYSCALE)

    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Visualizar keypoints
    img1_kp = cv2.drawKeypoints(img1, kp1, None, color=(0, 255, 0),
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    img2_kp = cv2.drawKeypoints(img2, kp2, None, color=(0, 255, 0),
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    axes[row, 0].imshow(img1_kp, cmap='gray')
    axes[row, 0].set_title(f'{os.path.basename(p1)} — {len(kp1)} keypoints')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(img2_kp, cmap='gray')
    axes[row, 1].set_title(f'{os.path.basename(p2)} — {len(kp2)} keypoints')
    axes[row, 1].axis('off')

    resultados_orb.append({
        "Par": titulo,
        "Img 1": os.path.basename(p1),
        "Keypoints 1": len(kp1),
        "Img 2": os.path.basename(p2),
        "Keypoints 2": len(kp2)
    })

    print(f"{titulo}:")
    print(f"  {os.path.basename(p1)}: {len(kp1)} keypoints | {os.path.basename(p2)}: {len(kp2)} keypoints")

plt.suptitle('Keypoints ORB Extraídos por Par de Imagens', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

df_orb = pd.DataFrame(resultados_orb)
display(df_orb)


### 3.3 Matching — BFMatcher


In [ ]:
# Matching com BFMatcher (Hamming para ORB)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

fig, axes = plt.subplots(3, 1, figsize=(18, 20))

resultados_match = []

for idx, (p1, p2, titulo) in enumerate(pares_orb):
    img1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(p2, cv2.IMREAD_GRAYSCALE)

    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Filtrar bons matches (distância < 50)
    bons_matches = [m for m in matches if m.distance < 50]

    # Exibir os 30 melhores matches
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2, matches[:30], None,
                                   flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    axes[idx].imshow(img_matches)
    axes[idx].set_title(f'{titulo}\nTotal matches: {len(matches)} | Bons (d<50): {len(bons_matches)}',
                        fontsize=11)
    axes[idx].axis('off')

    dist_media = np.mean([m.distance for m in matches]) if matches else 0

    resultados_match.append({
        "Par": titulo,
        "Total Matches": len(matches),
        "Bons Matches (d<50)": len(bons_matches),
        "Distância Média": round(dist_media, 2)
    })

    print(f"{titulo}: {len(matches)} matches total, {len(bons_matches)} bons (d<50), dist. média = {dist_media:.2f}")

plt.suptitle('Matching ORB com BFMatcher (Hamming)', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

df_match = pd.DataFrame(resultados_match)
display(df_match)


### Discussão do Matching

**Houve mais correspondências úteis entre imagens da mesma classe?**
- Sim. Os pares intra-classe (Flores–Flores e Folhas–Folhas) tendem a apresentar maior quantidade de bons matches e menores distâncias médias, pois compartilham padrões visuais similares (texturas de pétalas, nervuras).

**Quantidade de matches significou qualidade?**
- Não necessariamente. O par inter-classes (Flores vs. Folhas) pode apresentar um número significativo de matches totais, mas com distâncias maiores — indicando correspondências forçadas e pouco confiáveis. A qualidade (distância baixa) é mais relevante que a quantidade bruta.

**Iluminação, escala, fundo ou ângulo atrapalharam?**
- Sim. Variações de iluminação e ângulo entre as capturas introduzem inconsistências nos descritores, reduzindo a qualidade dos matches. O fundo desfocado (bokeh) gera keypoints espúrios que não correspondem ao objeto de interesse.


### 3.4 Teste de Robustez


In [ ]:
# Teste de robustez: original vs versões modificadas
img_teste = cv2.imread(imgs_A[0], cv2.IMREAD_GRAYSCALE)

# Gerar versões modificadas
# 1. Rotação leve (15°)
h, w = img_teste.shape
M_rot = cv2.getRotationMatrix2D((w // 2, h // 2), 15, 1.0)
img_rotacionada = cv2.warpAffine(img_teste, M_rot, (w, h))

# 2. Redução de contraste (multiplicar por 0.5)
img_baixo_contraste = np.clip(img_teste.astype(np.float32) * 0.5 + 50, 0, 255).astype(np.uint8)

# 3. Borramento com Gaussiano
img_borrada = cv2.GaussianBlur(img_teste, (15, 15), sigmaX=5)

versoes = [
    ("Original", img_teste),
    ("Rotação 15°", img_rotacionada),
    ("Baixo Contraste", img_baixo_contraste),
    ("Borramento Gaussiano 15×15", img_borrada)
]

resultados_robustez = []
bf_robust = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

kp_orig, des_orig = orb.detectAndCompute(img_teste, None)

for i, (nome, img_v) in enumerate(versoes):
    kp_v, des_v = orb.detectAndCompute(img_v, None)

    if i == 0:
        n_matches, n_bons, dist_m = "-", "-", "-"
    else:
        matches = bf_robust.match(des_orig, des_v)
        matches = sorted(matches, key=lambda x: x.distance)
        bons = [m for m in matches if m.distance < 50]
        n_matches = len(matches)
        n_bons = len(bons)
        dist_m = round(np.mean([m.distance for m in matches]), 2) if matches else 0

    img_kp = cv2.drawKeypoints(img_v, kp_v, None, color=(0, 255, 0))
    axes[i].imshow(img_kp, cmap='gray')
    axes[i].set_title(f'{nome}\n{len(kp_v)} keypoints')
    axes[i].axis('off')

    resultados_robustez.append({
        "Versão": nome,
        "Keypoints": len(kp_v),
        "Total Matches": n_matches,
        "Bons (d<50)": n_bons,
        "Dist. Média": dist_m
    })

plt.suptitle(f'Teste de Robustez ORB — {os.path.basename(imgs_A[0])}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

df_robustez = pd.DataFrame(resultados_robustez)
display(df_robustez)


### Discussão do Teste de Robustez

**O número de keypoints mudou?**
- Sim. O borramento reduz significativamente o número de keypoints detectados, pois suaviza as bordas e gradientes que o ORB utiliza para detectar cantos (baseado em FAST). A rotação leve manteve aproximadamente o mesmo número de keypoints. O baixo contraste também tende a reduzir a detecção.

**A qualidade dos matches caiu?**
- A rotação de 15° manteve boa qualidade de correspondência, demonstrando a invariância rotacional do ORB. Já o borramento e a redução de contraste reduziram a qualidade dos matches (maiores distâncias de Hamming), pois alteram o padrão local de intensidade utilizado pelo descritor BRIEF (base do ORB).

**Até que ponto o descritor local se mostrou robusto?**
- O ORB demonstrou boa robustez para rotações leves e moderada robustez para variações de contraste. No entanto, é sensível a borramento forte, que destrói as estruturas de alta frequência que os descritores locais capturam. Para cenários com borramento severo, pré-processamento com sharpening ou o uso de descritores mais robustos (SIFT, SURF) seria recomendável.


---
## Parte 4 — Análise de Textura: GLCM e LBP

### 4.1 Seleção das Regiões

Selecionamos 1 imagem de cada classe, convertida para escala de cinza com quantização a 64 níveis para a GLCM, e utilizamos recortes centrais representativos da superfície do objeto para focar na textura.


### 4.2 GLCM — Matriz de Coocorrência de Níveis de Cinza


In [ ]:
# GLCM — 1 imagem de cada classe
from skimage.feature import graycomatrix, graycoprops

def analisar_glcm(img_path, classe_nome, ax_img, niveis=64):
    # Calcula GLCM e extrai propriedades para uma imagem.
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    # Quantização para 'niveis' tons
    img_q = (img // (256 // niveis)).astype(np.uint8)

    # Recorte central (patch representativo)
    h, w = img_q.shape
    ch, cw = h // 4, w // 4
    patch = img_q[ch:h-ch, cw:w-cw]

    # GLCM com 1 distância e 4 ângulos (0°, 45°, 90°, 135°)
    angulos = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    nomes_ang = ['0°', '45°', '90°', '135°']
    glcm = graycomatrix(patch, distances=[1], angles=angulos,
                        levels=niveis, symmetric=True, normed=True)

    # Extrair propriedades
    props = {}
    for prop_name in ['contrast', 'homogeneity', 'energy', 'correlation']:
        valores = graycoprops(glcm, prop_name)[0]
        for j, ang in enumerate(nomes_ang):
            props[f'{prop_name}_{ang}'] = round(valores[j], 4)

    # Visualização do patch e da GLCM (ângulo 0°)
    ax_img.imshow(patch, cmap='gray')
    ax_img.set_title(f'{classe_nome}\nPatch Central')
    ax_img.axis('off')

    return props, patch

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

props_A, patch_A = analisar_glcm(imgs_A[0], 'Classe A (Flores)', axes[0])
props_B, patch_B = analisar_glcm(imgs_B[0], 'Classe B (Folhas)', axes[1])

plt.suptitle('Patches Centrais Utilizados na Análise GLCM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabela comparativa GLCM
dados_glcm = []
for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
    for ang in ['0°', '45°', '90°', '135°']:
        chave = f'{prop}_{ang}'
        dados_glcm.append({
            "Propriedade": prop.capitalize(),
            "Ângulo": ang,
            "Classe A (Flores)": props_A[chave],
            "Classe B (Folhas)": props_B[chave]
        })

df_glcm = pd.DataFrame(dados_glcm)
print("=" * 80)
print("TABELA DE PROPRIEDADES GLCM")
print("=" * 80)
display(df_glcm)


### Interpretação da GLCM

| Propriedade | O que mede | Flores (Classe A) | Folhas (Classe B) |
|-------------|-----------|-------------------|-------------------|
| **Contrast** | Variação local de intensidade | Tende a ser menor (pétalas suaves) | Tende a ser maior (nervuras criam transições) |
| **Homogeneity** | Uniformidade local | Mais alta (textura suave) | Mais baixa (padrões lineares variados) |
| **Energy** | Uniformidade da distribuição da GLCM | Mais alta (menos padrões distintos) | Mais baixa (mais diversidade textural) |
| **Correlation** | Correlação linear entre vizinhos | Alta em ambas as classes | Ligeiramente diferente nos ângulos alinhados com nervuras |

**Observação angular:** Nas folhas, espera-se anisotropia — as propriedades variam com o ângulo porque as nervuras possuem orientação preferencial. Nas flores, a textura é mais isotrópica (sem direção preferencial clara).


### 4.3 LBP — Local Binary Pattern


In [ ]:
# LBP — 1 imagem de cada classe
from skimage.feature import local_binary_pattern

LBP_RADIUS = 2
LBP_POINTS = 8 * LBP_RADIUS  # 16 pontos

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (path, classe) in enumerate([(imgs_A[0], "A (Flores)"), (imgs_B[0], "B (Folhas)")]):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Recorte central
    h, w = img.shape
    ch, cw = h // 4, w // 4
    patch = img[ch:h-ch, cw:w-cw]

    # Calcular LBP
    lbp = local_binary_pattern(patch, LBP_POINTS, LBP_RADIUS, method='uniform')

    # Histograma normalizado do LBP
    n_bins = LBP_POINTS + 2  # bins uniformes + 1 não-uniforme
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)

    # Exibir patch original
    axes[row, 0].imshow(patch, cmap='gray')
    axes[row, 0].set_title(f'Classe {classe} — Patch Original')
    axes[row, 0].axis('off')

    # Exibir mapa LBP
    axes[row, 1].imshow(lbp, cmap='hot')
    axes[row, 1].set_title(f'Classe {classe} — Mapa LBP')
    axes[row, 1].axis('off')

    # Exibir histograma LBP
    cor = '#e74c3c' if row == 0 else '#27ae60'
    axes[row, 2].bar(range(n_bins), lbp_hist, color=cor, alpha=0.8, edgecolor='black')
    axes[row, 2].set_title(f'Classe {classe} — Histograma LBP')
    axes[row, 2].set_xlabel('Padrão LBP (uniforme)')
    axes[row, 2].set_ylabel('Frequência Normalizada')
    axes[row, 2].grid(True, alpha=0.4)

plt.suptitle(f'Análise LBP (R={LBP_RADIUS}, P={LBP_POINTS}, método uniforme)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Discussão do LBP

**O LBP capturou microestruturas visuais úteis?**
- Sim. O mapa LBP das folhas revelou os padrões lineares das nervuras de forma mais marcante (padrões de borda no LBP), enquanto as flores apresentaram um mapa LBP mais homogêneo, refletindo a textura suave e uniforme das pétalas.

**As classes ficaram mais separáveis por textura?**
- Os histogramas LBP mostraram perfis distintos entre flores e folhas. As folhas tendem a ter maior proporção de padrões de borda (bins intermediários do histograma uniforme), enquanto as flores concentram mais padrões uniformes (flat patterns, bins extremos). Essa diferença, embora sutil, é discriminativa.

**GLCM e LBP trouxeram evidências coerentes entre si?**
- Sim. Ambos os descritores confirmam que as folhas possuem textura mais contrastada e anisotrópica (nervuras lineares), enquanto as flores possuem textura mais suave e isotrópica. A GLCM capturou essa diferença via contrast/homogeneity, e o LBP via distribuição de padrões de borda vs. uniformes.


---
## Parte 5 — Síntese Integradora

### Tabela Comparativa Final

| Tipo de Descritor | O que Representa | Vantagens | Limitações | Funcionou bem no meu dataset? |
|---|---|---|---|---|
| **Histogramas** | Distribuição global de intensidade/cor dos pixels | Rápido de calcular; invariante a posição; captura a paleta cromática da cena | Perde informação espacial; sensível a iluminação; não distingue texturas | ✅ **Sim** — A dominância do canal R (flores) vs. G (folhas) é altamente discriminativa |
| **Estatística Global** (média, variância, entropia) | Resumo numérico da distribuição de intensidades | Compacto; fácil de comparar; entropia indica complexidade da imagem | Muito resumido; duas imagens muito diferentes podem ter mesma média | ⚠️ **Parcial** — Entropia e variância diferem entre classes, mas com sobreposição |
| **ORB + Matching** | Regiões locais de interesse e suas correspondências entre imagens | Invariante a rotação; rápido; bom para verificar similaridade estrutural | Sensível a borramento e compressão; matches inter-classe podem ser ruidosos | ⚠️ **Parcial** — Mais matches intra-classe, mas variação de iluminação/fundo gera ruído |
| **GLCM** | Relações espaciais entre pixels vizinhos (textura) | Captura textura direcional; várias propriedades interpretáveis | Requer quantização prévia; sensível a escala do patch; custo computacional | ✅ **Sim** — Nervuras das folhas geram anisotropia e contraste que diferenciam das flores |
| **LBP** | Microestruturas de textura local (padrões binários) | Invariante a iluminação monotônica; compacto; rápido | Sensível a ruído; raio fixo limita a escala captada | ✅ **Sim** — Perfis de histograma LBP distintos entre pétalas suaves e nervuras lineares |


### Conclusão Final

#### 1. Qual descritor foi mais útil no seu dataset?
O **histograma de cor RGB** (análise global) foi o descritor mais imediatamente discriminativo, pois a diferença cromática entre flores (predominância de vermelho/rosa) e folhas (predominância de verde) é a característica visual mais forte do dataset. A diferença nos canais de cor é tão marcante que um classificador simples baseado na razão R/G já separaria as classes com alta acurácia.

Em segundo lugar, os **descritores de textura (GLCM e LBP)** forneceram informação complementar valiosa, capturando a diferença entre a textura suave das pétalas e a textura linear/nervurada das folhas.

#### 2. Qual descritor foi menos útil?
A **estatística global** (média, variância, desvio padrão) foi o descritor menos discriminativo isoladamente, pois resume excessivamente a imagem em poucos números. Duas imagens de classes diferentes podem ter médias e variâncias similares se tiverem distribuições de intensidade sobrepostas. A entropia foi ligeiramente mais informativa, mas ainda insuficiente para separação confiável.

#### 3. Em que situações a aquisição influenciou fortemente os resultados?
- **Iluminação:** Imagens capturadas sob luz direta forte apresentaram histogramas mais deslocados para tons claros, independentemente da classe, reduzindo o poder discriminativo da análise global.
- **Distância/Ângulo:** Close-ups revelaram texturas finas (nervuras, estames) que beneficiaram GLCM/LBP, enquanto planos gerais diluíram esses detalhes.
- **Fundo desfocado (bokeh):** A abertura f/1.9 criou desfoques que geraram keypoints ORB espúrios no background, prejudicando o matching.

#### 4. Se eu fosse treinar um sistema simples de Visão Computacional com esse dataset, quais características usaria primeiro?
Combinaria, em ordem de prioridade:
1. **Histogramas de cor RGB normalizados** — como feature principal, pela alta capacidade discriminativa entre as classes;
2. **Propriedades GLCM** (contrast, homogeneity) — como features complementares para capturar diferenças texturais;
3. **Histograma LBP normalizado** — para robustez adicional contra variações de iluminação, pois o LBP é parcialmente invariante a mudanças monotônicas de brilho.

Essa combinação de descritores globais (cor) + texturais (GLCM + LBP) criaria um vetor de características multidimensional capaz de alimentar com eficácia classificadores como SVM, Random Forest ou KNN.
